In [ ]:
import pandas                  as pd
import numpy                   as np
import seaborn                 as sns
import matplotlib.pyplot       as plt
import seaborn                 as sns
import tensorflow              as tf
import tensorflow.keras        as keras
import heapq


from math                      import sqrt
from tensorflow.keras.models   import Sequential
from tensorflow.keras.layers   import Dense
from tensorflow.keras.layers   import Flatten
from tensorflow.keras.layers   import LSTM
from tensorflow.keras.layers   import RepeatVector
from tensorflow.keras.layers   import TimeDistributed
from sklearn.preprocessing     import LabelEncoder
from sklearn.preprocessing     import RobustScaler,MinMaxScaler,KBinsDiscretizer

import os

In [ ]:
def rmsle(y, y_):
    log1 = np.nan_to_num(np.array([np.log(v + 1) for v in y]))
    log2 = np.nan_to_num(np.array([np.log(v + 1) for v in y_]))
    calc = (log1 - log2) ** 2
    return np.sqrt(np.mean(calc))

In [ ]:
def downcast_dtypes(df):
    float_cols = [c for c in df if df[c].dtype == "float64"]
    int_cols = [c for c in df if df[c].dtype in ["int64", "int32"]]
    df[float_cols] = df[float_cols].astype(np.float32)
    df[int_cols] = df[int_cols].astype(np.int16)
    return df

In [ ]:
def to_supervise(X,y,n_input,n_output):
    X_out, y_out = list(), list()
    for _ in range(len(X)):
        in_end = in_start + n_input
        out_end = in_end + n_output
        if (out_end <= len(X)):
            X_out.append(X[in_start:in_end])
            y_out.append(y[in_end:out_end])
        in_start += 1
    return array(X_out,y_out)

In [ ]:
def rmsle(y_true,y_pred):
    msle = keras.losses.MeanSquaredLogarithmicError()
    return keras.sqrt(msle(y_true, y_pred)) 

In [ ]:
def build_model(X,y,n_input,n_output):
    X, y = to_supervise(X,y,n_input,n_output)
    verbose, epoches, batch_size = 0, 50 ,16
    n_timesteps, n_features, n_outputs = n_input, train_x.shape[2], n_output
    # reshape output into [samples, timesteps, features]
    y = y.reshape((y.shape[0], y.shape[1], 1))
    # define model
    model = Sequential()
    model.add(LSTM(200, activation='relu', input_shape=(n_timesteps, n_features)))
    model.add(RepeatVector(n_outputs))
    model.add(LSTM(200, activation='relu', return_sequences=True))
    model.add(TimeDistributed(Dense(100, activation='relu')))
    model.add(TimeDistributed(Dense(1)))
    
    optimizer = keras.optimizers.RMSprop(learning_rate=0.001)
    model.compile(loss=rmsle, optimizer=optimizer)
    # fit network
    model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=verbose)
    return model

In [ ]:
def forecast(model, X, y, n_input, n_output):
    # flatten data
    data = array(history)
    data = data.reshape((data.shape[0]*data.shape[1], data.shape[2]))
    # retrieve last observations for input data
    input_x = data[-n_input:, :]
    # reshape into [1, n_input, n]
    input_x = input_x.reshape((1, input_x.shape[0], input_x.shape[1]))
    # forecast the next week
    yhat = model.predict(input_x, verbose=0)
    # we only want the vector forecast
    yhat = yhat[0]
    return yhat

In [ ]:
holiday = pd.read_csv("../input/store-sales-time-series-forecasting/holidays_events.csv")
oil  = pd.read_csv("../input/store-sales-time-series-forecasting/oil.csv")
stores = pd.read_csv("../input/store-sales-time-series-forecasting/stores.csv")
transactions = pd.read_csv("../input/store-sales-time-series-forecasting/transactions.csv")
train = pd.read_csv("../input/store-sales-time-series-forecasting/train.csv")
test = pd.read_csv("../input/store-sales-time-series-forecasting/test.csv")
sample_submission = pd.read_csv("../input/store-sales-time-series-forecasting/sample_submission.csv")

In [ ]:
#investigating the files
files = [holiday,oil,stores,transactions,train,test,sample_submission]
titles = ["Holiday","Oil","Stores","Transactions","Train","Test","Sample_submission"]

for file, title in zip(files,titles):
    print("##################")
    print(title)
    print(file.columns)
    #print(file)

In [ ]:
for file, title in zip(files,titles):
    print("##################")
    print(title)
    object_cols = []
    number_cols = []
    for col in file.columns:
        if (file[col].dtype == 'object'):
            object_cols.append(col)
        else:
            number_cols.append(col)
    print("{} Object Columns".format(title))
    print(object_cols)
    print("{} Numerical Columns".format(title))
    print(number_cols)
    #print(file[number_cols])
    for col in object_cols:
        print("Unique values in column {} of {}:".format(col,file))
        print(file[col].value_counts())
        #print("Number of unique values".format(len(file[col].unique())))
        print("******************")

In [ ]:
train['store_nbr'] = train['store_nbr'].astype(str)
test['store_nbr'] = test['store_nbr'].astype(str)

In [ ]:
train_sales = train.pivot(index="date", columns=['family','store_nbr'], values="sales")
train_promotions = train.pivot(index="date", columns=['family','store_nbr'], values="onpromotion")
test_promotions = test.pivot(index="date", columns=['family','store_nbr'], values="onpromotion")

In [ ]:
train_sales.columns = ['_'.join(col) for col in train_sales.columns.values]
train_sales.columns = ['Sales '+ col for col in train_sales.columns.values]
train_promotions.columns = ['_'.join(col) for col in train_promotions.columns.values]
train_promotions.columns = ['Promotions '+ col for col in train_promotions.columns.values]
test_promotions.columns = ['_'.join(col) for col in test_promotions.columns.values]
test_promotions.columns = ['Promotions '+ col for col in test_promotions.columns.values]

In [ ]:
X_train = train_promotions
y_train = train_sales
X_test  = test_promotions

In [ ]:
## combine datasets
X_train = X_train.merge(holiday, on = 'date', how='left')
X_train = X_train.merge(oil, on = 'date', how='left')
train = train.merge(stores, on = 'store_nbr', how='left')
train = train.merge(transactions, on = ['date', 'store_nbr'], how='left')
print(train.columns)
train = train.rename(columns = {"type_x" : "holiday_type", "type_y" : "store_type"})

test = test.merge(holiday, on = 'date', how='left')
test = test.merge(oil, on = 'date', how='left')
test = test.merge(stores, on = 'store_nbr', how='left')
test = test.merge(transactions, on = ['date', 'store_nbr'], how='left')
test = test.rename(columns = {"type_x" : "holiday_type", "type_y" : "store_type"})

In [ ]:
train = downcast_dtypes(train)
test  = downcast_dtypes(test)

In [ ]:
print(len(train[train.date=='2013-05-02'].store_nbr.unique()))
print(len(train[train.date=='2013-05-02'].family.unique()))
print(train[train.date=='2013-05-02'].family.unique())
print(train[train.date=='2013-01-01'].family.unique())

In [ ]:
train['date_parsed'] = pd.to_datetime(train['date'],format="%Y-%m-%d")
not_datetime = pd.to_datetime(train['date'], errors='coerce',format="%Y-%m-%d").isna()
print("Rows in training data with wrong date and time format:")
print(not_datetime[not_datetime == True])

In [ ]:
test['date_parsed'] = pd.to_datetime(test['date'],format="%Y-%m-%d")
not_datetime = pd.to_datetime(test['date'], errors='coerce',format="%Y-%m-%d").isna()
print("Rows in training data with wrong date and time format:")
print(not_datetime[not_datetime == True])

In [ ]:
train['quarter']    = train['date_parsed'].dt.quarter
train['month']      = train['date_parsed'].dt.month
train['year']       = train['date_parsed'].dt.year
train['dayofyear']  = train['date_parsed'].dt.dayofyear
train['weekofyear'] = train['date_parsed'].dt.isocalendar().week

In [ ]:
test['quarter']    = test['date_parsed'].dt.quarter
test['month']      = test['date_parsed'].dt.month
test['year']       = test['date_parsed'].dt.year
test['dayofyear']  = test['date_parsed'].dt.dayofyear
test['weekofyear'] = test['date_parsed'].dt.isocalendar().week

In [ ]:
train['date_parsed'] = train['date_parsed'].values.astype(float)
test['date_parsed'] = test['date_parsed'].values.astype(float)

In [ ]:
print(train.date_parsed)

In [ ]:
print("Train columns:")
print(train.columns)

In [ ]:
train[['store_nbr', 'cluster']] = train[['store_nbr', 'cluster']].astype(str)
test[['store_nbr', 'cluster']] = test[['store_nbr', 'cluster']].astype(str)

In [ ]:
## Item Categories object and number columns
object_cols = []
number_cols = []
for col in train.columns:
    if (train[col].dtype == 'object'):
        object_cols.append(col)
    else:
        number_cols.append(col)
print("Training Object Columns")
print(object_cols)
print("Training Numerical Columns")
print(number_cols)

In [ ]:
agg = train.groupby('date').agg({"sales" : "mean"}).reset_index()
fig = agg.plot(kind="line", x="date", y="sales",title = "Average Sales by Date")

agg = train.groupby('date').agg({"transactions" : "mean"}).reset_index()
fig = agg.plot(kind="line", x="date", y="transactions",title = "Average Transactions by Date")

In [ ]:
agg = train.groupby(['date', 'store_type']).agg({"sales" : "mean"}).reset_index()
agg.pivot(columns='store_type').interpolate().plot()

agg = train.groupby(['date', 'store_type']).agg({"transactions" : "mean"}).reset_index()
agg.pivot(columns='store_type').interpolate().plot()

In [ ]:
train.drop(['date'],axis=1,inplace=True)
test.drop(['date'], axis=1,inplace=True)

In [ ]:
y = train['sales']
X = train
X.drop(['sales'],axis=1,inplace=True) 

In [ ]:
train[['holiday_type','locale','locale_name','description']] = train[['holiday_type','locale','locale_name','description']].fillna("Work Day")
test[['holiday_type','locale','locale_name','description']] = test[['holiday_type','locale','locale_name','description']].fillna("Work Day")
train.loc[train['transferred'] == True,['holiday_type','locale','locale_name','description']] = "Work Day"
test.loc[test['transferred'] == True,['holiday_type','locale','locale_name','description']] = "Work Day"

train['transferred'] = train['transferred'].fillna(True)
test['transferred'] = test['transferred'].fillna(True)
train['dcoilwtico'] = train['dcoilwtico'].fillna((train['dcoilwtico'].mean()), inplace=False)
test['dcoilwtico'] = test['dcoilwtico'].fillna((test['dcoilwtico'].mean()), inplace=False)

In [ ]:
missing_data_count = train.isnull().sum()
print("Missing Data in Training Set:")
print(missing_data_count)
sns.heatmap(train.isnull())
plt.show()

In [ ]:
missing_data_count = test.isnull().sum()
print("Missing Data in Test Set:")
print(missing_data_count)
sns.heatmap(test.isnull())
plt.show()

In [ ]:
XandTest = pd.concat([X,test])
s = (X.dtypes == 'object')
object_cols = list(s[s].index)
print("Object Columns")
print(object_cols)
label_encoder = LabelEncoder()
for col in object_cols:
    if (col == 'date_parsed'):
        continue
    XandTest[col] = label_encoder.fit_transform(XandTest[col])
    X[col] = label_encoder.transform(X[col])
    test[col]  = label_encoder.transform(test[col])

In [ ]:
columns  = X.columns
pipe     = Pipeline([("scaler_1", RobustScaler()),("scaler_2", MinMaxScaler())])
XandTest = pd.concat([X,test])
XandTest = pipe.fit(XandTest)
X        = pipe.transform(X)
test     = pipe.transform(test)
X        = pd.DataFrame(X,   columns=columns)
test     = pd.DataFrame(test,columns=columns)

In [ ]:
param_grid = {'n_estimators': [2000,5000,10000],
             'learning_rate': [0.01,0.05,0.1]}
print("Parameter grid:\n{}".format(param_grid))

In [ ]:
X_train = X[:2748913]
y_train = y[:2748913]
X_test  = X[2748913:]
y_test  = y[2748913:]

In [ ]:
RMSLE = metrics.make_scorer(rmsle, greater_is_better=False)
xgb = XGBRegressor(tree_method='gpu_hist')
xgb.fit(X_train,y_train)
y_pred_train = xgb.predict(X_train)
y_pred_test  = xgb.predict(X_test)

In [ ]:
error = rmsle(y_pred_test,y_test)
print(error)

In [ ]:
y_pred_train = pd.Series(y_pred_train)
y_pred_test = pd.Series(y_pred_test,index=range(len(X_train),len(X_train)+len(X_test)))
print(y_pred_test)

In [ ]:
X_train['sales'] = y_train
X_train['sales_pred'] = y_pred_train
X_test['sales'] = y_test
X_test['sales_pred'] = y_pred_test

In [ ]:
agg_train_sales = X_train.groupby('date_parsed').agg({"sales" : "mean"}).reset_index()
agg_train_sales_pred = X_train.groupby('date_parsed').agg({"sales_pred" : "mean"}).reset_index()
agg_test_sales = X_test.groupby('date_parsed').agg({"sales" : "mean"}).reset_index()
agg_test_sales_pred = X_test.groupby('date_parsed').agg({"sales_pred" : "mean"}).reset_index()

In [ ]:
plt.figure()
plt.plot(range(len(agg_train_sales)), agg_train_sales['sales'],color='blue', label="train")
#plt.plot(range(len(agg_train_sales),len(agg_train_sales)+len(agg_test_sales)), agg_test_sales['sales'], '-', color='red',label="test")
plt.plot(range(len(agg_train_sales_pred)), agg_train_sales_pred['sales_pred'], '--', color='green',label="prediction train")
#plt.plot(range(len(agg_train_sales_pred),len(agg_train_sales_pred)+len(agg_test_sales_pred)), agg_test_sales_pred['sales_pred'], '--',color='green',label="prediction test")
plt.legend(loc=(1.01, 0))
plt.xlabel("Date")
plt.ylabel("Sales")
#plt.ylim(0,30000)
plt.show()

In [ ]:
score_y, y_pred = grid_search_cv(X,y,test,param_grid)
print("Score is:", score_y)

In [ ]:
test_set = pd.read_csv("../input/store-sales-time-series-forecasting/test.csv")
output  = pd.DataFrame({'id': test_set.id, 'sales': y_pred})
output.to_csv('my_submission_code.csv',index=False)